In [2]:
import re
import os
import pandas as pd
import numpy as np
from pathlib import PurePath, PurePosixPath
from h_anonypy.modules_dicom import get_folder_list, replace_digits, check_dicom_from_folder
from h_anonypy.modules_dicom import get_patient_info, anonymize_dicom_file

from pydicom.errors import InvalidDicomError
def check_dcm_from_folder(root_dir, return_list=False):
    

    dcm_dir_list = []
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file != "DICOMDIR" and not file.startswith('._'):
                filepath = os.path.join(dirpath, file)
                try:
                    pydicom.dcmread(filepath, stop_before_pixels=True)
                    if return_list == True:
                        dcm_dir_list.append(dirpath)
                        break
                    else:
                        return filepath
                except InvalidDicomError:
                    continue
                except Exception:
                    continue
    if return_list == True:
        return dcm_dir_list
    else:
        return False

def get_meta_info(dicom_file):
    ds = pydicom.dcmread(dicom_file)
    ds.SpecificCharacterSet = 'ISO_IR 192'  # UTF-8
    # ds.SpecificCharacterSet = 'ISO_IR 149'  # EUC-KR
    ds.decode()
    info = {
        'PatientID':    ds.get('PatientID'),
        'PatientName':  str(ds.get('PatientName')),
        'PatientSex':   ds.get('PatientSex'),
        'PatientAge':   ds.get('PatientAge'),
        'PatientBirthDate': ds.get('PatientBirthDate'),
        'AcquisitionDate': ds.get('AcquisitionDate', 'No AcquisitionDate'),
        'PatientSize': ds.get('PatientSize'),
        'PatientWeight': ds.get('PatientWeight'),
        'OtherPatientIDs': ds.get('OtherPatientIDs'),
        'OtherPatientNames': str(ds.get('OtherPatientNames')),
        'InstitutionName': ds.get('InstitutionName'),
        'ReferringPhysicianName': str(ds.get('ReferringPhysicianName')),
        'AccessionNumber': ds.get('AccessionNumber'),
        'Modality': ds.get('Modality'),
        'BodyPartExamined': ds.get('BodyPartExamined'),
        'Manufacturer': str(ds.get('Manufacturer')),
        'ManufacturerModelName': str(ds.get('ManufacturerModelName'))

    }
    return info



In [93]:
# 3569 Surggram
# 4092 CT
# 3169 RE

# EMR = pd.read_excel('../emr/data/통합_RAW_V2.0.0_1987-2023_250929.xlsx')

# IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
UGI_META = IMAGE_META['UGI'].copy()
UGI_ADD = pd.read_csv('./data/UGI_20250624.csv')
UGI_ADD['Center'] = 'SEVERANCE'
UGI_ADD['ImportDate'] = 20250624
len(UGI_ADD['hutom_id'].unique())

UGI_IDs = pd.read_csv('./data/UGI_MissingID_Mapping.csv')
UGI_IDs.loc[len(UGI_IDs),['old_hutom_id','new_hutom_id']] = ['UGI05028','UGI0299']
UGI_IDs

# UGI_ADD[~(UGI_ADD['hutom_id'].isin(UGI_IDs['old_hutom_id'].unique().tolist()))]
# UGI_ADD

# import_idx = UGI_META[UGI_META['ImportDate']==20250624].index.tolist()
# UGI_META.drop(import_idx, axis=0, inplace=True)
# UGI_META


# sample_list = os.listdir('/nas/nas6/UGI/CT_SurgGram')
# test_ct[~test_ct['hutom_id'].isin(sample_list)]
# len(sample_list)

# test_ct[test_ct.duplicated(subset=['hutom_id'], keep=False)].loc[:300]

# UGI05028
# UGI0299



,old_hutom_id,new_hutom_id,new_patient_id
0,UGI03396,UGI1975,1854885.0
1,UGI03397,UGI2611,1952602.0
2,UGI03398,UGI2592,170118.0
3,UGI03399,UGI0483,1902529.0
4,UGI03400,STEMR7383,1913022.0
...,...,...,...
3165,UGI06562,UGI3215,10912072.0
3166,UGI06563,UGI3218,10913366.0
3167,UGI06564,UGI3219,10913902.0
3168,UGI06565,UGI3221,10916129.0


In [243]:
from tqdm.notebook import tqdm
import sys
from pathlib import Path

sys.path.append('/home/yhchoi/PROJECT/Pneumo')
from pneumo_reconpy import (ImageReader,
                            AbdomenSegmenter, 
                            UmbilicusDetector, 
                            predictUmbilicus)

from pneumo_reconpy import (sitk_normalize_array, 
                            sitk_transform_and_resample,
                            sitk_compute_rotation)

# IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
# VIDEO_META = pd.read_excel('./SHEET/VIDEO_META.xlsx', sheet_name=None)
# HUTOM_ID = pd.read_excel('./SHEET/HUTOM_ID.xlsx', sheet_name=None)

UGI_META = IMAGE_META['UGI'].copy()
UGI_META_CT = UGI_META[UGI_META['Modality']=='CT'].copy()
UGI_META_CT = UGI_META_CT[~(UGI_META_CT['Center']=='GradientHealth')].reset_index(drop=True)

select = UGI_META_CT[UGI_META_CT['AcquisitionDate']=='19000101'].reset_index(drop=True)
select['N.slices'] = 0
select['Thickness'] = 0

hid_list = select['hutom_id'].tolist()
base_dir = Path('/nas/nas6/UGI/CT_SurgGram')
for i in tqdm(range(len(hid_list))):

    sample_folder = base_dir / hid_list[i] / '00_DICOM'

    sample_volumes = []
    sample_thcknesses = []
    sample_paths = []
    for dirpath, _, filenames in os.walk(sample_folder):
        for filename in filenames:
            if '_iso.' not in filename:
                sample_dir = Path(dirpath) / filename
                sample_volume = ImageReader(sample_dir).sitk_volume
                sample_volumes.append(sample_volume)
                sample_thcknesses.append(sample_volume.GetSpacing()[2])
                sample_paths.append(Path(dirpath))

    # slicethickness가 가장 얇은 볼륨 선택
    idx = np.argmin(sample_thcknesses)
    sample_dir = sample_paths[idx]
    sample_volume = sample_volumes[idx]
    select.loc[i,'N.slices'] = sample_volume.GetDepth()
    select.loc[i,'Thickness'] = sample_thcknesses[idx]

select



  0%|          | 0/296 [00:00<?, ?it/s]

/tmp/ipykernel_2013381/1383801250.py:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9977578520774841' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  select.loc[i,'Thickness'] = sample_thcknesses[idx]


KeyboardInterrupt: 

In [269]:

for i in tqdm(range(138,len(hid_list))):

    try:
        sample_folder = base_dir / hid_list[i] / '00_DICOM'

        sample_volumes = []
        sample_thcknesses = []
        sample_paths = []
        for dirpath, _, filenames in os.walk(sample_folder):
            for filename in filenames:
                if '_iso.' not in filename:
                    sample_dir = Path(dirpath) / filename
                    sample_volume = ImageReader(sample_dir).sitk_volume
                    sample_volumes.append(sample_volume)
                    sample_thcknesses.append(sample_volume.GetSpacing()[2])
                    sample_paths.append(Path(dirpath))

        # slicethickness가 가장 얇은 볼륨 선택
        idx = np.argmin(sample_thcknesses)
        sample_dir = sample_paths[idx]
        sample_volume = sample_volumes[idx]
        select.loc[i,'N.slices'] = sample_volume.GetDepth()
        select.loc[i,'Thickness'] = sample_thcknesses[idx]
    except:
        print(i,hid_list[i])
        pass

select




  0%|          | 0/158 [00:00<?, ?it/s]

138 UGI0281
144 UGI0288
145 UGI0289
160 UGI0308
161 UGI0309
167 UGI0315
171 UGI0319
172 UGI0320
174 UGI0325
186 UGI0339
211 UGI0368
217 UGI0376
222 UGI0383
224 UGI0385
228 UGI0389
229 UGI0390
236 UGI0399
239 UGI0402
241 UGI0404
245 UGI0408
246 UGI0409
249 UGI0412
260 UGI0425
268 UGI0435
269 UGI0436
273 UGI0440
281 UGI0449
284 UGI0453
293 UGI3261
294 UGI3262


,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,...,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate,N.slices,Thickness
0,UGI0028,ANONYMIZE_0028,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,671997,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0028\1,SEVERANCE,20240108,446,0.997758
1,UGI0029,ANONYMIZE_0029,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672007,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0029\1,SEVERANCE,20240108,454,0.997797
2,UGI0030,ANONYMIZE_0030,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672002,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0030\1,SEVERANCE,20240108,518,0.998070
3,UGI0031,ANONYMIZE_0031,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672004,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0031\1,SEVERANCE,20240108,468,0.997863
4,UGI0032,ANONYMIZE_0032,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672001,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0032\1,SEVERANCE,20240108,474,0.997890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,UGI0498,8215853,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672133,CT,NaN,UGI\[CT]신촌세브란스 501\8215853\1,SEVERANCE,20240108,437,0.997712
292,UGI0499,8244550,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672142,CT,NaN,UGI\[CT]신촌세브란스 501\8244550\1,SEVERANCE,20240108,495,0.997980
293,UGI3261,3885391,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672050,CT,NaN,UGI\[CT]신촌세브란스 501\3885391\1,SEVERANCE,20240108,0,0.000000
294,UGI3262,8120826,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672080,CT,NaN,UGI\[CT]신촌세브란스 501\8120826\1,SEVERANCE,20240108,0,0.000000


In [270]:
# i=60
# sample_folder = base_dir / hid_list[i] / '00_DICOM'

# sample_volumes = []
# sample_thcknesses = []
# sample_paths = []
# for dirpath, _, filenames in os.walk(sample_folder):
#     for filename in filenames:
#         if '_iso.' not in filename:
#             sample_dir = Path(dirpath) / filename
#             print(sample_dir)

select


,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,...,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate,N.slices,Thickness
0,UGI0028,ANONYMIZE_0028,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,671997,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0028\1,SEVERANCE,20240108,446,0.997758
1,UGI0029,ANONYMIZE_0029,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672007,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0029\1,SEVERANCE,20240108,454,0.997797
2,UGI0030,ANONYMIZE_0030,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672002,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0030\1,SEVERANCE,20240108,518,0.998070
3,UGI0031,ANONYMIZE_0031,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672004,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0031\1,SEVERANCE,20240108,468,0.997863
4,UGI0032,ANONYMIZE_0032,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672001,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0032\1,SEVERANCE,20240108,474,0.997890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,UGI0498,8215853,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672133,CT,NaN,UGI\[CT]신촌세브란스 501\8215853\1,SEVERANCE,20240108,437,0.997712
292,UGI0499,8244550,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672142,CT,NaN,UGI\[CT]신촌세브란스 501\8244550\1,SEVERANCE,20240108,495,0.997980
293,UGI3261,3885391,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672050,CT,NaN,UGI\[CT]신촌세브란스 501\3885391\1,SEVERANCE,20240108,0,0.000000
294,UGI3262,8120826,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,UNKNOWN,UNKNOWN,672080,CT,NaN,UGI\[CT]신촌세브란스 501\8120826\1,SEVERANCE,20240108,0,0.000000


In [271]:
from tqdm.notebook import tqdm

select_col = ['hutom_id','PatientID','PatientSex','PatientAge','PatientBirthDate','AcquisitionDate','AccessionNumber','folder']

# UGI_META_CT = UGI_META[UGI_META['Modality']=='CT'].copy()
# UGI_META_CT = UGI_META_CT[~(UGI_META_CT['Center']=='GradientHealth')].reset_index(drop=True)
UGI_META_CT = select.copy()

ids = UGI_META_CT['hutom_id'].unique().tolist()
print(len(ids))

col_meta = ['new_unit','sex','age','opyear','opdate','birthday','pre_wt','pre_ht','pre_BMI']
UGI_META_CT[col_meta] = None
for i in tqdm(range(len(ids))):
    
    sample = UGI_META_CT[UGI_META_CT['hutom_id']==ids[i]]
    sample_demo = db_raw_emr_demo[db_raw_emr_demo['hutom_id']==ids[i]]
    if len(sample_demo) == 0 and sample['PatientID'].isna().sum() == 0:
        sample_demo = db_raw_emr_demo[db_raw_emr_demo['hutom_id'].isin(sample['PatientID'].tolist())]

    if len(sample_demo) > 0:
        UGI_META_CT.loc[sample.index,col_meta] = sample_demo[col_meta].values
    
UGI_META_CT.to_excel('data/CT_META_20251215_NoDate.xlsx', index=None)
UGI_META_CT


# UGI_META_CT['opdate'] = pd.to_datetime(UGI_META_CT['opdate'], errors='coerce')
# UGI_META_CT['ctdate'] = pd.to_datetime(UGI_META_CT['AcquisitionDate'], errors='coerce')
# UGI_META_CT['ct_pod'] = (UGI_META_CT['ctdate'] - UGI_META_CT['opdate']).dt.days
# UGI_META_CT['pre_wt'] = UGI_META_CT['pre_wt'].replace('UK', 0)
# UGI_META_CT['pre_ht'] = UGI_META_CT['pre_ht'].replace('UK', 0)
# UGI_META_CT['pre_BMI'] = UGI_META_CT['pre_BMI'].replace('UK', 0)
# wt = UGI_META_CT['pre_wt'].astype(float).values
# ht = UGI_META_CT['pre_ht'].astype(float).values
# UGI_META_CT['pre_BSA'] = np.sqrt((wt*ht)/3600)





296


  0%|          | 0/296 [00:00<?, ?it/s]

,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,...,Thickness,new_unit,sex,age,opyear,opdate,birthday,pre_wt,pre_ht,pre_BMI
0,UGI0028,ANONYMIZE_0028,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997758,7978703,M,68,2015,2015-01-21 00:00:00,1946-02-25 00:00:00,60,170,20.8
1,UGI0029,ANONYMIZE_0029,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997797,7978973,M,52,2015,2015-01-22 00:00:00,1962-05-29 00:00:00,55,168,19.5
2,UGI0030,ANONYMIZE_0030,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.998070,4383374,M,56,2015,2015-01-23 00:00:00,1958-05-30 00:00:00,79,169,27.7
3,UGI0031,ANONYMIZE_0031,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997863,3978355,F,79,2015,2015-01-28 00:00:00,1935-02-08 00:00:00,66.8,152.4,28.8
4,UGI0032,ANONYMIZE_0032,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997890,5300491,F,66,2015,2015-01-28 00:00:00,1948-06-03 00:00:00,48.6,153.5,20.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,UGI0498,8215853,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997712,8215853,F,70,2016,2016-04-18 00:00:00,1945-11-16 00:00:00,43.3,158.8,17.2
292,UGI0499,8244550,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.997980,8244550,M,44,2016,2016-05-26 00:00:00,1972-01-10 00:00:00,73,178,23
293,UGI3261,3885391,ANONYMIZE,M,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.000000,None,None,None,None,None,None,None,None,None
294,UGI3262,8120826,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,...,0.000000,None,None,None,None,None,None,None,None,None


In [ ]:
##### INPUT #####
# IMAGE_META : 전체 영상 메타정보 
# HUTOM_ID : 전체 hutom id 리스트
# dicom_dir : 반입 데이터 경로
# center: 반입 기관
# importdate: 반입 날짜
# organ: 조직 정보
# n_digits: hutom id 자리수

IMAGE_META = pd.read_excel('..\\NEW_DB\\IMAGE_META.xlsx', sheet_name=None)
HUTOM_ID = pd.read_excel('..\\NEW_DB\\HUTOM_ID.xlsx', sheet_name=None)
dicom_dir = ['w:\\DataTeam\\TEST']
center = ['CENTER']
importdate = ['20250000']
organ = ['LIVER']
n_digits = 4


In [ ]:
## Start

sample_info_list = []
for i in range(len(dicom_dir)):

    # 1. 입력 폴더 내 샘플 폴더 리스트
    folder_list = get_folder_list(dicom_dir[i])
    for j in range(len(folder_list)):

        # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
        check_path = os.path.join(dicom_dir[i], folder_list[j])
        dcm_fname = check_dicom_from_folder(check_path)
        if dcm_fname:
            metadata = get_patient_info(dcm_fname)

            anony_folder = re.sub(r'[가-힣]+', '', folder_list[j])
            anony_folder = re.sub(r'\d+', replace_digits, anony_folder)
            anony_folder = re.sub(r'\s+', '', anony_folder)

            posix = PurePath(dcm_fname)
            fpath = PurePosixPath(organ[i], *posix.parts[2:-1])
            metadata['folder'] = fpath
            metadata['Center'] = center[i]
            metadata['ImportDate'] = importdate[i]
            
            sample_info_list.append(metadata)

    # 2. 샘플 메타 [반입리스트에서 중복 제거 ? default=False]
    sample_info = pd.DataFrame.from_dict(sample_info_list)
    check_dup_col = ['PatientID','PatientName','AcquisitionDate','PatientSex']
    # sample_info = sample_info.drop_duplicates(subset=check_dup_col, ignore_index=True)
    sample_info.insert(0, 'hutom_id', None)

    # 3. 중복 제거 및 HUTOM ID 부여
    meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1

    sample_ids = sample_info['PatientID'].unique().tolist()
    for idx in range(len(sample_ids)):
        dup = meta_all[meta_all['PatientID']==sample_ids[idx]]
        if len(dup) == 0:
            hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
            id_number += 1
        elif len(dup) > 0:
            hutomid = dup["hutom_id"].tolist()[0]
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
        else:
            print('check sample')

    # 4. Anonymous
    anonymous_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')
    sample_ids = sample_info["PatientID"].unique().tolist()
    for idx in range(len(sample_ids)):
        # 4.1. 샘플 선택
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]

        # 4.2. 폴더/경로, 영상 종류, HUTOM ID
        sample_path = check_sample["folder"].tolist()
        hutomid = check_sample['hutom_id'].tolist()    
        for m in range(len(sample_path)):
            # 샘플 기준, 하위 폴더 탐색
            posix = PurePath(sample_path[m])
            dcm_path = os.path.join(dicom_dir[i], posix.parts[2])

            dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)
            for n in range(len(dcm_folder_list)):
                # 하위 폴더 기준, 익명화: Name, ID > Hutom ID
                raw_dir = dcm_folder_list[n]
                no_ko_folder = re.sub(r'[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7A3]+', '', 
                                    posix.parts[2]).strip()

                save_dir = os.path.join(anonymous_dir, hutomid[m], 
                                        no_ko_folder, os.path.join(*posix.parts[3:]))

                os.makedirs(save_dir, exist_ok=True)

                for dirpath, _, filenames in os.walk(raw_dir):
                    for filename in filenames:
                        anonymize_dicom_file(raw_dir, filename, save_dir, id=hutomid[m])

    # 5. Add information > check !!!!
    ids_add = pd.DataFrame(sample_info['hutom_id'].unique().tolist(),
                        columns=['hutom_id'])
    ids_add['dicom'] = 'O'
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["dicom"] = ids_all_add["dicom_x"].fillna(ids_all_add["dicom_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]
    meta_all_add = pd.concat([meta_all, sample_info], ignore_index=True)

##### OUTPUT > IMAGE_META, HUTOM_ID 업데이트 및 저장
# IMAGE_META[organ[i]]
# meta_all_add




In [4]:
# checking manufacture
from tqdm.notebook import tqdm
import pydicom
pydicom.config.convert_wrong_length_to_UN = True

# dicom_dir = ['/nas/nas6/URO/CT',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01014_서울아산병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01011_신촌세브란스병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01016_전남화순대병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01002_아주대병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01020_삼성서울병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01028_칠곡경북대병원',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/00_2025/01004_구리한양대병원'
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01014_서울아산',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01015_강남세브',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01011_신촌세브',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01020_삼성서울',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01012_서울성모',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01016_화순전남',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01025_서울대',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01026_노원을지',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01003_성빈센트',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01028_칠곡경북대',
#              '/nas/nas3/Service/RUS_Kidney/Modeling/01004_한양대구리병원']
dicom_dir = ['/nas/nas6/URO/CT-FDA']

## Start
sample_info_list = []
for i in range(len(dicom_dir)):

    # 1. 입력 폴더 내 샘플 폴더 리스트
    folder_list = get_folder_list(dicom_dir[i])
    for j in tqdm(range(len(folder_list))):

        # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
        check_path = os.path.join(dicom_dir[i], folder_list[j])
        dcm_fname = check_dcm_from_folder(check_path)
        if dcm_fname:
            metadata = get_meta_info(dcm_fname)
            sample_info_list.append(metadata)

        else:
            print(check_path)

sample_info = pd.DataFrame.from_dict(sample_info_list)
sample_info



  0%|          | 0/140 [00:00<?, ?it/s]

,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,OtherPatientNames,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,Manufacturer,ManufacturerModelName
0,UROFDA1,GRDN0BQBJDWFHN3F,M,062Y,19611022,20231025,NaN,NaN,None,None,xenops-995729-K81AZ,ANON NAME,GRDN255CVIS56AGU,CT,Chest/Abdomen,FUJI,SCENARIA View
1,UROFDA40,GRDN1LKR9C8LEGG6,F,043Y,19660820,20100507,NaN,NaN,None,None,thryothor-495511-FAQ3N,ANON NAME,GRDNARL8OSYZEWBH,CT,Chest/Abdomen,Philips,Brilliance 64
2,UROFDA2,GRDN2D7Q7IZIY90W,M,068Y,19540519,20230322,NaN,100.000000,None,None,rustica-743701-BO4Y1,ANON NAME,GRDNLFT1CCD5P9X5,CT,None,Philips,Brilliance 64
3,UROFDA3,GRDN48AQDOUFML9T,F,None,19661227,No AcquisitionDate,NaN,NaN,None,None,rustica-743701-other,ANON NAME,GRDNYPQY0EAT9T9U,CT,Other,Unknown,Unknown
4,UROFDA4,GRDN50FLU1TU8XXG,F,054Y,19690808,20231114,NaN,NaN,None,None,auritus-681591-J1073,ANON NAME,GRDNBABFDRBB124P,CT,Abdomen,SIEMENS,SOMATOM go.Now
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,UROFDA87,UROFDA87,F,054Y,19661224,20211213,NaN,NaN,None,None,auritus-681591-UJAM3,ANON NAME,GRDN2ILMCUQTJZ5X,CT,Chest/Abdomen,TOSHIBA,Aquilion Lightning
136,UROFDA88,UROFDA88,F,057Y,19650110,20220215,1.4986,81.192968,None,None,quelea-19938-1XIAI,ANON NAME,GRDNF41EGXQ4JQY7,CT,Abdomen,SIEMENS,SOMATOM go.Up
137,UROFDA89,UROFDA89,F,066Y,19560630,20221203,1.6764,92.532768,None,None,quelea-19938-LZPAG,ANON NAME,GRDNLPDUTQLFDWXR,CT,Abdomen,SIEMENS,SOMATOM go.Up
138,UROFDA90,UROFDA90,F,079Y,19431107,20230404,1.6890,74.390000,ANON,ANON NAME,thryothor-495511-K61QM,ANON NAME,GRDNHFZONJPZLNR4,CT,Chest,SIEMENS,SOMATOM go.All


,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,OtherPatientNames,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate
601,UGI0288,ANONYMIZE_0288,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672171,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0288\1,SEVERANCE,20240108
602,UGI0288,ANONYMIZE_0288,ANONYMIZE,F,0Y,19000101.0,19000101,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672171,SR,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0288\997,SEVERANCE,20240108
